In [2]:
!pip -q install transformers datasets evaluate accelerate sentencepiece rouge_score sacrebleu

from google.colab import drive
drive.mount('/content/drive')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.5 MB/s eta 0:00:00
Mounted at /content/drive


In [3]:
!pip -q install transformers datasets evaluate accelerate sentencepiece rouge_score sacrebleu

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datasets import Dataset

PROJECT_DIR = Path("/content/drive/MyDrive/Senior2_Medical_Captioning")
MASTER_PATH = PROJECT_DIR / "master_caption_concepts_scored.csv"
LABEL_VOCAB_PATH = Path("/content/drive/MyDrive/ImageCLEF/artifacts_consultive_swin/label_vocab.json")

master_df = pd.read_csv(MASTER_PATH)

with open(LABEL_VOCAB_PATH, "r") as f:
    label_vocab = json.load(f)

cui_list = label_vocab["labels"]

print("Master shape:", master_df.shape)
print(master_df["split"].value_counts())
print("Number of CUIs:", len(cui_list))

display(master_df.head())

Master shape: (116604, 20)
split
train    97364
valid    19240
Name: count, dtype: int64
Number of CUIs: 2646


,ID,reference_caption,License,Attribution,gt_CUIs,pred_CUIs_micro,pred_CUIs_coverage,split,image_filename,caption_image_zip_path,concept_image_zip_path,gt_CUIs_count,pred_CUIs_micro_count,pred_CUIs_coverage_count,pred_CUIs_micro_precision,pred_CUIs_micro_recall,pred_CUIs_micro_f1,pred_CUIs_coverage_precision,pred_CUIs_coverage_recall,pred_CUIs_coverage_f1
0,ImageCLEFmedical_Caption_2026_train_0,Head CT demonstrating left parotiditis.,CC BY,Peres et al.,C0040405,C0040405,C0040405,train,ImageCLEFmedical_Caption_2026_train_0.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,1,1,1,1.0,1.0,1.000000,1.0,1.0,1.000000
1,ImageCLEFmedical_Caption_2026_train_1,Chest X-ray showing enlarged cardiac silhouett...,CC BY-NC,Al Mulhim et al.,C1306645;C0817096;C0442800;C0018787;C0242073,C0817096;C1306645,C0817096;C1306645,train,ImageCLEFmedical_Caption_2026_train_1.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,5,2,2,1.0,0.4,0.571429,1.0,0.4,0.571429
2,ImageCLEFmedical_Caption_2026_train_2,CT chest axial view showing a huge ascending a...,CC BY-NC,Al Mulhim et al.,C0040405;C0856747,C0040405,C0040405,train,ImageCLEFmedical_Caption_2026_train_2.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,2,1,1,1.0,0.5,0.666667,1.0,0.5,0.666667
3,ImageCLEFmedical_Caption_2026_train_3,Acquired renal cysts in end-stage renal failur...,CC BY,Vester et al.,C0041618,C0041618,C0041618,train,ImageCLEFmedical_Caption_2026_train_3.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,1,1,1,1.0,1.0,1.000000,1.0,1.0,1.000000
4,ImageCLEFmedical_Caption_2026_train_4,Computed tomography (CT) shows floating thromb...,CC BY,Sato et al.,C0040405;C0040053,C0040405,C0040405,train,ImageCLEFmedical_Caption_2026_train_4.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,2,1,1,1.0,0.5,0.666667,1.0,0.5,0.666667


In [5]:
def clean_cuis(x):
    if pd.isna(x) or str(x).strip() == "":
        return []
    return [c.strip() for c in str(x).split(";") if c.strip()]

def make_cui_input(cui_string, source="ground-truth"):
    cuis = clean_cuis(cui_string)

    if len(cuis) == 0:
        cui_text = "<NO_CONCEPTS>"
    else:
        cui_text = " ".join([f"<{c}>" for c in cuis])

    return f"generate medical caption from {source} umls concepts: {cui_text}"

df = master_df.copy()

df["input_text"] = df["gt_CUIs"].apply(lambda x: make_cui_input(x, "ground-truth"))
df["target_text"] = df["reference_caption"].astype(str)

train_full_df = df[df["split"] == "train"][["ID", "input_text", "target_text"]].reset_index(drop=True)
valid_full_df = df[df["split"] == "valid"][["ID", "input_text", "target_text"]].reset_index(drop=True)

# For faster evaluation during training
valid_eval_df = valid_full_df.sample(n=3000, random_state=42).reset_index(drop=True)

print("Train full:", train_full_df.shape)
print("Valid full:", valid_full_df.shape)
print("Valid eval sample:", valid_eval_df.shape)

print("\nExample input:")
print(train_full_df.iloc[0]["input_text"])

print("\nTarget:")
print(train_full_df.iloc[0]["target_text"])

Train full: (97364, 3)
Valid full: (19240, 3)
Valid eval sample: (3000, 3)

Example input:
generate medical caption from ground-truth umls concepts: <C0040405>

Target:
Head CT demonstrating left parotiditis.


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

special_tokens = [f"<{cui}>" for cui in cui_list] + ["<NO_CONCEPTS>"]

num_added = tokenizer.add_tokens(special_tokens)
model.resize_token_embeddings(len(tokenizer))

print("Model:", MODEL_NAME)
print("Added tokens:", num_added)
print("Tokenizer size:", len(tokenizer))

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: CPU only")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model: t5-base
Added tokens: 2647
Tokenizer size: 34747
GPU: NVIDIA A100-SXM4-80GB


In [7]:
MAX_INPUT_LEN = 256
MAX_TARGET_LEN = 96

train_dataset = Dataset.from_pandas(train_full_df)
valid_dataset = Dataset.from_pandas(valid_eval_df)

def preprocess_function(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LEN,
        truncation=True
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LEN,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_valid = valid_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=valid_dataset.column_names
)

print(tokenized_train)
print(tokenized_valid)

Map:   0%|          | 0/97364 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 97364
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3000
})


In [8]:
import evaluate
import numpy as np

rouge = evaluate.load("rouge")
bleu = evaluate.load("sacrebleu")

def compute_metrics(eval_pred):
    preds, labels = eval_pred

    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.asarray(preds)
    labels = np.asarray(labels)

    # If predictions are logits, convert to token ids
    if preds.ndim == 3:
        preds = np.argmax(preds, axis=-1)

    vocab_size = len(tokenizer)

    # Replace invalid prediction token ids
    preds = np.where(
        (preds < 0) | (preds >= vocab_size),
        tokenizer.pad_token_id,
        preds
    )

    # Replace ignored label ids
    labels = np.where(
        labels != -100,
        labels,
        tokenizer.pad_token_id
    )

    preds = preds.astype(np.int64)
    labels = labels.astype(np.int64)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    rouge_result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    bleu_result = bleu.compute(
        predictions=decoded_preds,
        references=[[l] for l in decoded_labels]
    )

    return {
        "rouge1": rouge_result["rouge1"],
        "rouge2": rouge_result["rouge2"],
        "rougeL": rouge_result["rougeL"],
        "bleu": bleu_result["score"]
    }

In [9]:
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from transformers.trainer_utils import get_last_checkpoint

OUTPUT_DIR = PROJECT_DIR / "models" / "gt_cui_t5_base_full"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=3e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,

    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,

    predict_with_generate=True,
    generation_max_length=96,
    generation_num_beams=2,

    fp16=True,
    logging_steps=200,
    report_to="none",

    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

last_checkpoint = get_last_checkpoint(str(OUTPUT_DIR))
print("Last checkpoint:", last_checkpoint)

if last_checkpoint is not None:
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    trainer.train()

Last checkpoint: /content/drive/MyDrive/Senior2_Medical_Captioning/models/gt_cui_t5_base_full/checkpoint-9129


[transformers] You are resuming training from a checkpoint trained with 5.12.0 of Transformers but your current version is 5.12.1. This is not recommended and could yield to errors or unwanted behaviors.
[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].
[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Epoch,Training Loss,Validation Loss


In [10]:
import json

metrics = trainer.evaluate()
print(metrics)

metrics_path = OUTPUT_DIR / "eval_metrics_valid3000.json"

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved:", metrics_path)

Training Loss,Validation Loss,Epoch,Rouge1,Rouge2,Rougel,Bleu
4.798126,2.340282,3,0.178668,0.051174,0.151620,2.403350


{'eval_loss': 2.340282440185547, 'eval_rouge1': 0.1786680949781121, 'eval_rouge2': 0.05117363996119807, 'eval_rougeL': 0.1516196705579745, 'eval_bleu': 2.4033501112786375}
Saved: /content/drive/MyDrive/Senior2_Medical_Captioning/models/gt_cui_t5_base_full/eval_metrics_valid3000.json


In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

sample_df = valid_eval_df.sample(n=8, random_state=7).reset_index(drop=True)

for i, row in sample_df.iterrows():
    input_text = row["input_text"]
    target_text = row["target_text"]

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LEN
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=96,
            num_beams=4,
            early_stopping=True
        )

    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("="*100)
    print("ID:", row["ID"])
    print("INPUT:", input_text)
    print("\nTARGET:", target_text)
    print("\nPRED:", pred)

ID: ImageCLEFmedical_Caption_2026_valid_10033
INPUT: generate medical caption from ground-truth umls concepts: <C1306645> <C0817096> <C0032227>

TARGET: Patient's chest X-ray one year prior to presentation The patient's previous chest X-ray showed no anomalies of acute significance. Please note the absence of pleural effusions in this chest X-ray.

PRED: Chest X-ray showing bilateral pleural effusions.
ID: ImageCLEFmedical_Caption_2026_valid_14934
INPUT: generate medical caption from ground-truth umls concepts: <C0040405> <C0006267>

TARGET: A low grade bronchiectasis without sign of GL-ILD in the left upper field (white arrow)

PRED: Computed tomography (CT) scan of the abdomen and pelvis with intravenous contrast.
ID: ImageCLEFmedical_Caption_2026_valid_16966
INPUT: generate medical caption from ground-truth umls concepts: <C0040405>

TARGET: Abdomen CT with contrast showing the presence of a soft tissue mass (red arrow).

PRED: Computed tomography (CT) scan of the abdomen and pelvis

In [12]:
from datasets import Dataset
import pandas as pd
import numpy as np

# نستخدم نفس validation sample المستخدم أثناء التدريب
valid_ids = valid_eval_df["ID"].tolist()

eval_base_df = master_df[master_df["ID"].isin(valid_ids)].copy()
eval_base_df["ID"] = pd.Categorical(eval_base_df["ID"], categories=valid_ids, ordered=True)
eval_base_df = eval_base_df.sort_values("ID").reset_index(drop=True)

print("Eval base:", eval_base_df.shape)
display(eval_base_df.head())

Eval base: (3000, 20)


,ID,reference_caption,License,Attribution,gt_CUIs,pred_CUIs_micro,pred_CUIs_coverage,split,image_filename,caption_image_zip_path,concept_image_zip_path,gt_CUIs_count,pred_CUIs_micro_count,pred_CUIs_coverage_count,pred_CUIs_micro_precision,pred_CUIs_micro_recall,pred_CUIs_micro_f1,pred_CUIs_coverage_precision,pred_CUIs_coverage_recall,pred_CUIs_coverage_f1
0,ImageCLEFmedical_Caption_2026_valid_12658,Ultra-high-frequency ultrasound (48 MHz probe)...,CC BY,Nagy et al.,C0041618;C0182400;C0042449,C0041618,C0041618,valid,ImageCLEFmedical_Caption_2026_valid_12658.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,3,1,1,1.0,0.333333,0.500000,1.0,0.333333,0.500000
1,ImageCLEFmedical_Caption_2026_valid_17336,A midesophageal long axis view zoomed up on th...,CC BY,Anand et al.,C0041618;C0003501;C0549113,C0041618,C0041618,valid,ImageCLEFmedical_Caption_2026_valid_17336.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,3,1,1,1.0,0.333333,0.500000,1.0,0.333333,0.500000
2,ImageCLEFmedical_Caption_2026_valid_18790,General aspect of the uterus and of the cervix...,CC BY,Manea et al.,C0041618;C0042149;C0007874,C0041618,C0005682;C0041618,valid,ImageCLEFmedical_Caption_2026_valid_18790.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,3,1,2,1.0,0.333333,0.500000,0.5,0.333333,0.400000
3,ImageCLEFmedical_Caption_2026_valid_12815,computed tomography of the chest with a white ...,CC BY,Karnan et al.,C0040405;C0817096,C0040405,C0040405,valid,ImageCLEFmedical_Caption_2026_valid_12815.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,2,1,1,1.0,0.500000,0.666667,1.0,0.500000,0.666667
4,ImageCLEFmedical_Caption_2026_valid_12666,Lateral view of the skull showing multiple pun...,CC BY,Muacevic et al.,C1306645;C0037303,C0037303;C1306645,C0037303;C1306645,valid,ImageCLEFmedical_Caption_2026_valid_12666.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,2,2,2,1.0,1.000000,1.000000,1.0,1.000000,1.000000


In [13]:
def make_cui_input_same_prompt(cui_string):
    cuis = clean_cuis(cui_string)

    if len(cuis) == 0:
        cui_text = "<NO_CONCEPTS>"
    else:
        cui_text = " ".join([f"<{c}>" for c in cuis])

    # نفس prompt التدريب تماماً
    return f"generate medical caption from ground-truth umls concepts: {cui_text}"

def build_eval_variant(base_df, cui_col):
    temp = base_df.copy()
    temp["input_text"] = temp[cui_col].apply(make_cui_input_same_prompt)
    temp["target_text"] = temp["reference_caption"].astype(str)
    return temp[["ID", "input_text", "target_text"]].reset_index(drop=True)

valid_gt_eval = build_eval_variant(eval_base_df, "gt_CUIs")
valid_micro_eval = build_eval_variant(eval_base_df, "pred_CUIs_micro")
valid_coverage_eval = build_eval_variant(eval_base_df, "pred_CUIs_coverage")

print("GT:", valid_gt_eval.shape)
print("Micro:", valid_micro_eval.shape)
print("Coverage:", valid_coverage_eval.shape)

GT: (3000, 3)
Micro: (3000, 3)
Coverage: (3000, 3)


In [14]:
def tokenize_eval_df(eval_df):
    dataset = Dataset.from_pandas(eval_df)
    tokenized = dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=dataset.column_names
    )
    return tokenized

tokenized_valid_gt = tokenize_eval_df(valid_gt_eval)
tokenized_valid_micro = tokenize_eval_df(valid_micro_eval)
tokenized_valid_coverage = tokenize_eval_df(valid_coverage_eval)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [15]:
gt_metrics = trainer.evaluate(eval_dataset=tokenized_valid_gt, metric_key_prefix="gt")
micro_metrics = trainer.evaluate(eval_dataset=tokenized_valid_micro, metric_key_prefix="pred_micro")
coverage_metrics = trainer.evaluate(eval_dataset=tokenized_valid_coverage, metric_key_prefix="pred_coverage")

comparison = pd.DataFrame([
    {
        "setting": "GT concepts",
        "loss": gt_metrics.get("gt_loss"),
        "rouge1": gt_metrics.get("gt_rouge1"),
        "rouge2": gt_metrics.get("gt_rouge2"),
        "rougeL": gt_metrics.get("gt_rougeL"),
        "bleu": gt_metrics.get("gt_bleu"),
    },
    {
        "setting": "Pred micro concepts",
        "loss": micro_metrics.get("pred_micro_loss"),
        "rouge1": micro_metrics.get("pred_micro_rouge1"),
        "rouge2": micro_metrics.get("pred_micro_rouge2"),
        "rougeL": micro_metrics.get("pred_micro_rougeL"),
        "bleu": micro_metrics.get("pred_micro_bleu"),
    },
    {
        "setting": "Pred coverage concepts",
        "loss": coverage_metrics.get("pred_coverage_loss"),
        "rouge1": coverage_metrics.get("pred_coverage_rouge1"),
        "rouge2": coverage_metrics.get("pred_coverage_rouge2"),
        "rougeL": coverage_metrics.get("pred_coverage_rougeL"),
        "bleu": coverage_metrics.get("pred_coverage_bleu"),
    },
])

display(comparison)

comparison_path = OUTPUT_DIR / "gt_vs_pred_eval_comparison_valid3000.csv"
comparison.to_csv(comparison_path, index=False)
print("Saved:", comparison_path)

Training Loss,Validation Loss,Epoch,Rouge1,Rouge2,Rougel,Bleu
4.798126,2.340282,3,0.178668,0.051174,0.151620,2.403350


Training Loss,Validation Loss,Epoch,Micro Loss,Micro Rouge1,Micro Rouge2,Micro Rougel,Micro Bleu
4.798126,No log,3,2.397974,0.202647,0.062125,0.171798,1.927309


Training Loss,Validation Loss,Epoch,Coverage Loss,Coverage Rouge1,Coverage Rouge2,Coverage Rougel,Coverage Bleu
4.798126,No log,3,2.393247,0.198436,0.060312,0.167863,1.928968


,setting,loss,rouge1,rouge2,rougeL,bleu
0,GT concepts,2.340282,0.178668,0.051174,0.151620,2.403350
1,Pred micro concepts,2.397974,0.202647,0.062125,0.171798,1.927309
2,Pred coverage concepts,2.393247,0.198436,0.060312,0.167863,1.928968


Saved: /content/drive/MyDrive/Senior2_Medical_Captioning/models/gt_cui_t5_base_full/gt_vs_pred_eval_comparison_valid3000.csv


In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

sample_ids = valid_eval_df.sample(n=5, random_state=11)["ID"].tolist()

for sample_id in sample_ids:
    row = master_df[master_df["ID"] == sample_id].iloc[0]

    inputs_dict = {
        "GT": make_cui_input_same_prompt(row["gt_CUIs"]),
        "Pred micro": make_cui_input_same_prompt(row["pred_CUIs_micro"]),
        "Pred coverage": make_cui_input_same_prompt(row["pred_CUIs_coverage"]),
    }

    print("="*120)
    print("ID:", sample_id)
    print("TARGET:", row["reference_caption"])
    print("GT CUIs:", row["gt_CUIs"])
    print("Pred micro CUIs:", row["pred_CUIs_micro"])
    print("Pred coverage CUIs:", row["pred_CUIs_coverage"])

    for name, input_text in inputs_dict.items():
        inputs = tokenizer(
            input_text,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_INPUT_LEN
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=96,
                num_beams=4,
                early_stopping=True
            )

        pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"\n{name} PRED:", pred)

ID: ImageCLEFmedical_Caption_2026_valid_4390
TARGET: Example of an anterior–posterior radiograph of the pelvis with calculation of the migration percentage (MP): MP = A/B × 100. A represents the portion of ossified femoral head laying lateral to Perkin's line (vertical line drawn through the lateral acetabular margin and perpendicular to Hilgenreiner's line, which passes through the superior aspect of the triradiate cartilage). B represents the whole ossified femoral head.
GT CUIs: C1306645;C0023216;C0030797;C0015813;C0007301
Pred micro CUIs: C0030797;C1306645
Pred coverage CUIs: C0015813;C0030797;C1306645

GT PRED: Chest computed tomography (CT) scan of a patient with COVID-19 pneumonia. Axial CT image of a patient with COVID-19 pneumonia shows a large right-sided pleural effusion (red arrow) and a small left-sided pleural effusion (blue arrow).

Pred micro PRED: Axial T1-weighted MRI of the brain showing a hyperintense lesion in the left frontal lobe.

Pred coverage PRED: Computed to

In [17]:
from datasets import Dataset
import pandas as pd
import numpy as np

# نستخدم نفس validation sample المستخدم أثناء التدريب
valid_ids = valid_eval_df["ID"].tolist()

eval_base_df = master_df[master_df["ID"].isin(valid_ids)].copy()
eval_base_df["ID"] = pd.Categorical(eval_base_df["ID"], categories=valid_ids, ordered=True)
eval_base_df = eval_base_df.sort_values("ID").reset_index(drop=True)

print("Eval base:", eval_base_df.shape)
display(eval_base_df.head())

Eval base: (3000, 20)


,ID,reference_caption,License,Attribution,gt_CUIs,pred_CUIs_micro,pred_CUIs_coverage,split,image_filename,caption_image_zip_path,concept_image_zip_path,gt_CUIs_count,pred_CUIs_micro_count,pred_CUIs_coverage_count,pred_CUIs_micro_precision,pred_CUIs_micro_recall,pred_CUIs_micro_f1,pred_CUIs_coverage_precision,pred_CUIs_coverage_recall,pred_CUIs_coverage_f1
0,ImageCLEFmedical_Caption_2026_valid_12658,Ultra-high-frequency ultrasound (48 MHz probe)...,CC BY,Nagy et al.,C0041618;C0182400;C0042449,C0041618,C0041618,valid,ImageCLEFmedical_Caption_2026_valid_12658.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,3,1,1,1.0,0.333333,0.500000,1.0,0.333333,0.500000
1,ImageCLEFmedical_Caption_2026_valid_17336,A midesophageal long axis view zoomed up on th...,CC BY,Anand et al.,C0041618;C0003501;C0549113,C0041618,C0041618,valid,ImageCLEFmedical_Caption_2026_valid_17336.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,3,1,1,1.0,0.333333,0.500000,1.0,0.333333,0.500000
2,ImageCLEFmedical_Caption_2026_valid_18790,General aspect of the uterus and of the cervix...,CC BY,Manea et al.,C0041618;C0042149;C0007874,C0041618,C0005682;C0041618,valid,ImageCLEFmedical_Caption_2026_valid_18790.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,3,1,2,1.0,0.333333,0.500000,0.5,0.333333,0.400000
3,ImageCLEFmedical_Caption_2026_valid_12815,computed tomography of the chest with a white ...,CC BY,Karnan et al.,C0040405;C0817096,C0040405,C0040405,valid,ImageCLEFmedical_Caption_2026_valid_12815.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,2,1,1,1.0,0.500000,0.666667,1.0,0.500000,0.666667
4,ImageCLEFmedical_Caption_2026_valid_12666,Lateral view of the skull showing multiple pun...,CC BY,Muacevic et al.,C1306645;C0037303,C0037303;C1306645,C0037303;C1306645,valid,ImageCLEFmedical_Caption_2026_valid_12666.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,2,2,2,1.0,1.000000,1.000000,1.0,1.000000,1.000000


In [18]:
def make_cui_input_same_prompt(cui_string):
    cuis = clean_cuis(cui_string)

    if len(cuis) == 0:
        cui_text = "<NO_CONCEPTS>"
    else:
        cui_text = " ".join([f"<{c}>" for c in cuis])

    # نفس prompt التدريب تماماً
    return f"generate medical caption from ground-truth umls concepts: {cui_text}"

def build_eval_variant(base_df, cui_col):
    temp = base_df.copy()
    temp["input_text"] = temp[cui_col].apply(make_cui_input_same_prompt)
    temp["target_text"] = temp["reference_caption"].astype(str)
    return temp[["ID", "input_text", "target_text"]].reset_index(drop=True)

valid_gt_eval = build_eval_variant(eval_base_df, "gt_CUIs")
valid_micro_eval = build_eval_variant(eval_base_df, "pred_CUIs_micro")
valid_coverage_eval = build_eval_variant(eval_base_df, "pred_CUIs_coverage")

print("GT:", valid_gt_eval.shape)
print("Micro:", valid_micro_eval.shape)
print("Coverage:", valid_coverage_eval.shape)

GT: (3000, 3)
Micro: (3000, 3)
Coverage: (3000, 3)


In [19]:
def tokenize_eval_df(eval_df):
    dataset = Dataset.from_pandas(eval_df)
    tokenized = dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=dataset.column_names
    )
    return tokenized

tokenized_valid_gt = tokenize_eval_df(valid_gt_eval)
tokenized_valid_micro = tokenize_eval_df(valid_micro_eval)
tokenized_valid_coverage = tokenize_eval_df(valid_coverage_eval)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [20]:
gt_metrics = trainer.evaluate(eval_dataset=tokenized_valid_gt, metric_key_prefix="gt")
micro_metrics = trainer.evaluate(eval_dataset=tokenized_valid_micro, metric_key_prefix="pred_micro")
coverage_metrics = trainer.evaluate(eval_dataset=tokenized_valid_coverage, metric_key_prefix="pred_coverage")

comparison = pd.DataFrame([
    {
        "setting": "GT concepts",
        "loss": gt_metrics.get("gt_loss"),
        "rouge1": gt_metrics.get("gt_rouge1"),
        "rouge2": gt_metrics.get("gt_rouge2"),
        "rougeL": gt_metrics.get("gt_rougeL"),
        "bleu": gt_metrics.get("gt_bleu"),
    },
    {
        "setting": "Pred micro concepts",
        "loss": micro_metrics.get("pred_micro_loss"),
        "rouge1": micro_metrics.get("pred_micro_rouge1"),
        "rouge2": micro_metrics.get("pred_micro_rouge2"),
        "rougeL": micro_metrics.get("pred_micro_rougeL"),
        "bleu": micro_metrics.get("pred_micro_bleu"),
    },
    {
        "setting": "Pred coverage concepts",
        "loss": coverage_metrics.get("pred_coverage_loss"),
        "rouge1": coverage_metrics.get("pred_coverage_rouge1"),
        "rouge2": coverage_metrics.get("pred_coverage_rouge2"),
        "rougeL": coverage_metrics.get("pred_coverage_rougeL"),
        "bleu": coverage_metrics.get("pred_coverage_bleu"),
    },
])

display(comparison)

comparison_path = OUTPUT_DIR / "gt_vs_pred_eval_comparison_valid3000.csv"
comparison.to_csv(comparison_path, index=False)
print("Saved:", comparison_path)

Training Loss,Validation Loss,Epoch,Rouge1,Rouge2,Rougel,Bleu
4.798126,2.340282,3,0.178668,0.051174,0.151620,2.403350


Training Loss,Validation Loss,Epoch,Micro Loss,Micro Rouge1,Micro Rouge2,Micro Rougel,Micro Bleu
4.798126,No log,3,2.397974,0.202647,0.062125,0.171798,1.927309


Training Loss,Validation Loss,Epoch,Coverage Loss,Coverage Rouge1,Coverage Rouge2,Coverage Rougel,Coverage Bleu
4.798126,No log,3,2.393247,0.198436,0.060312,0.167863,1.928968


,setting,loss,rouge1,rouge2,rougeL,bleu
0,GT concepts,2.340282,0.178668,0.051174,0.151620,2.403350
1,Pred micro concepts,2.397974,0.202647,0.062125,0.171798,1.927309
2,Pred coverage concepts,2.393247,0.198436,0.060312,0.167863,1.928968


Saved: /content/drive/MyDrive/Senior2_Medical_Captioning/models/gt_cui_t5_base_full/gt_vs_pred_eval_comparison_valid3000.csv


In [21]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

sample_ids = valid_eval_df.sample(n=5, random_state=11)["ID"].tolist()

for sample_id in sample_ids:
    row = master_df[master_df["ID"] == sample_id].iloc[0]

    inputs_dict = {
        "GT": make_cui_input_same_prompt(row["gt_CUIs"]),
        "Pred micro": make_cui_input_same_prompt(row["pred_CUIs_micro"]),
        "Pred coverage": make_cui_input_same_prompt(row["pred_CUIs_coverage"]),
    }

    print("="*120)
    print("ID:", sample_id)
    print("TARGET:", row["reference_caption"])
    print("GT CUIs:", row["gt_CUIs"])
    print("Pred micro CUIs:", row["pred_CUIs_micro"])
    print("Pred coverage CUIs:", row["pred_CUIs_coverage"])

    for name, input_text in inputs_dict.items():
        inputs = tokenizer(
            input_text,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_INPUT_LEN
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=96,
                num_beams=4,
                early_stopping=True
            )

        pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"\n{name} PRED:", pred)

ID: ImageCLEFmedical_Caption_2026_valid_4390
TARGET: Example of an anterior–posterior radiograph of the pelvis with calculation of the migration percentage (MP): MP = A/B × 100. A represents the portion of ossified femoral head laying lateral to Perkin's line (vertical line drawn through the lateral acetabular margin and perpendicular to Hilgenreiner's line, which passes through the superior aspect of the triradiate cartilage). B represents the whole ossified femoral head.
GT CUIs: C1306645;C0023216;C0030797;C0015813;C0007301
Pred micro CUIs: C0030797;C1306645
Pred coverage CUIs: C0015813;C0030797;C1306645

GT PRED: Chest computed tomography (CT) scan of a patient with COVID-19 pneumonia. Axial CT image of a patient with COVID-19 pneumonia shows a large right-sided pleural effusion (red arrow) and a small left-sided pleural effusion (blue arrow).

Pred micro PRED: Axial T1-weighted MRI of the brain showing a hyperintense lesion in the left frontal lobe.

Pred coverage PRED: Computed to